In [4]:
import os
import mne
import yasa
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
from pathlib import Path
import re

Extraigo las señales ecg, falta procesarlas y extraer caracteristicas

In [5]:
def get_epochs(raw): 
    epochs = mne.make_fixed_length_epochs(
    raw,
    duration=10,   # segundos
    overlap=0,     # sin solapamiento
    preload=True
)
    


def cargar_registro(path,channels=None ):

   raw = mne.io.read_raw_edf(path, preload=True, verbose=False)   
   if channels != None:
      raw.pick(channels)
   raw.filter(0.3, 49, verbose=False)
   epochs = mne.make_fixed_length_epochs(
    raw,
    duration=10,   
    overlap=0,      
    preload=True,
    verbose=False
)

   data = epochs.get_data() * 1e6 #microVolts (porque mne trabaja en V) 
     

 
   return data



def crear_lista(ruta,channels=None, excluir=None ):
    if excluir is None:
        excluir=[]
    df=pd.DataFrame(columns=["case","subject" ,"data"])
    for archivo in ruta.glob("*.edf"): 
        if archivo.name not in excluir:#Esta linea es para filtrar los datos que no tienen un registro de emociones
            patron = r"case(\d+)_s(\d+)"
            match = re.search(patron, archivo.stem)
            
            if match:
                case_id = int(match.group(1))
                subject_id = int(match.group(2))
                
                data = cargar_registro(archivo,channels )
                X=[case_id,subject_id,data]
                df.loc[len(df)] = X
             
    return df

Channel names: 
['Fp1', 'Fp2', 'AF7', 'AF3', 'AF4', 'AF8', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2', 'EOG-HL', 'EOG-HR', 'EOG-VU', 'EOG-VD', 'EMG']

In [6]:
a=cargar_registro("REM_Turku/Data/PSG/case01_s10.edf")
 

Calcular potencia alfa en canales f3 y f4:

In [ ]:


def bandpower(signal, sf, band, n_fft=1024):
    """
    signal: 1D array del canal (n_muestras)
    sf: frecuencia de muestreo
    band: (fmin, fmax)
    """
    fmin, fmax = band
    
    psd, freqs = mne.time_frequency.psd_array_welch(
        signal,
        sfreq=sf,
        fmin=fmin,
        fmax=fmax,
        n_fft=n_fft,
        n_overlap=n_fft//2,
        verbose=False
    )
     
    power = np.trapezoid(psd, freqs)
    return power




def calc_FAA(df,band,sf):
    bands = {
        "delta": (0.5, 4),
        "theta": (4, 7),
        "alpha": (8, 12),
        "beta":  (13, 30),
        "gamma": (30, 45)
    }
 


    f3_dict = {}
    f4_dict = {}
    for i, row in df.iterrows():
        f3_dict[str(row["case"])] = []
        f4_dict[str(row["case"])] = [] 

        for epoch in row["data"]:
            F3_alpha = bandpower(epoch[0], sf, bands[band])
            F4_alpha = bandpower(epoch[1], sf, bands[band])
            f3_dict[str(row["case"])].append(F3_alpha)
            f4_dict[str(row["case"])].append(F4_alpha)
    return f3_dict, f4_dict

In [28]:
ruta = Path("REM_Turku/Data/PSG")
df1=pd.read_csv("REM_Turku/Records.csv")
excluir=df1[df1["Has more data"]==0]["Filename"].values#Lista con datos incompletos

preFAA=crear_lista(ruta,["F3","F4"],excluir)


f3,f4=calc_FAA(preFAA,"alpha",500)

 

In [29]:
f3

{'2': [np.float64(6.377150937801227),
  np.float64(6.6778379230846765),
  np.float64(8.241882138579964),
  np.float64(6.97164875358823),
  np.float64(11.660884093495893),
  np.float64(20.62187445666754),
  np.float64(6.177129723606743),
  np.float64(8.829507444725193),
  np.float64(12.04111328565764),
  np.float64(16.913867340228663),
  np.float64(9.865855004324374),
  np.float64(8.399036485894852)],
 '3': [np.float64(5.628785905377421),
  np.float64(6.437314628747258),
  np.float64(25.609007824529744),
  np.float64(8.661470482655801),
  np.float64(10.798543538937265),
  np.float64(7.224060847172418),
  np.float64(8.428584344749654),
  np.float64(8.694807475868448),
  np.float64(8.382311401967083),
  np.float64(6.686685649027692),
  np.float64(5.810479488908044),
  np.float64(5.863442253035879)],
 '4': [np.float64(6.7505971119692765),
  np.float64(6.86810305015025),
  np.float64(4.929070048330031),
  np.float64(4.6794606423019856),
  np.float64(6.156126093456017),
  np.float64(7.753493

In [21]:
def FAA(f3,f4):
    return np.log(f4) - np.log(f3)

In [22]:
faa_list=[]

for i in range(len(f3_list)):
    faa=FAA(f3_list[i],f4_list[i])
    faa_list.append(faa)

Ground truth:

In [ ]:
def clasificar_emociones(row):


    
    if row["pa_sum"] > 2 * row["na_sum"] and row["pa_sum"] > 5:
        return "POSITIVO" 
    else:
        return "NO POSITIVO"#Las emociones negativas se reportan mucho menos, en esta categoria la mayoria de sueños son "neutrales", es decir, los indices de emociones son muy bajos para considerarse positivos o negativos.
        #De las 53 filas que entran en no positivo, solo cuatro pasarian un filtro de este tipo: row["pa_sum"] > 2 * row["na_sum"] and row["pa_sum"] > 5 pero para emociones negativas

In [ ]:
 
df = pd.read_csv("REM_Turku/Data/Ratings.csv")


df=df[df["Filename"]!="case133_s27.edf"]#Este cambio es porque falta un edf en los datos
df.loc[df["Filename"] == 'case123_s27.edf', "Filename"] = 'case132_s27.edf'#Este cambio es por un typo en el csv
pa=[col for col in df.columns if col.startswith("SR_PA")] 
na=[col for col in df.columns if col.startswith("SR_NA")]  
df["pa_sum"] = df[pa].sum(axis=1)
df["na_sum"] = df[na].sum(axis=1)    

df['sentimiento'] = df.apply(clasificar_emociones, axis=1)



In [89]:
df[df["Filename"]=='case123_s27.edf']

,Filename,DreamReport_Wordcount,ER1_PA1,ER4_PA2,ER8_PA3,ER11_PA4,ER12_PA5,ER13_PA6,ER14_PA7,ER15_PA8,...,SR_NA5,SR_NA6,SR_NA7,SR_NA8,SR_NA9,SR_NA10,Remarks,pa_sum,na_sum,sentimiento
119,case123_s27.edf,231,0,0,0,0,0,0,0,0,...,1.0,0.0,0.0,0.0,0.0,0.0,NaN,18.0,2.0,POSITIVO


In [101]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [106]:
print(y)

[1 0 0 1 1 1 1 0 1 1 0 1 0 1 0 0 0 1 0 1 1 0 1 0 0 0 0 0 0 0 0 1 1 0 1 0 0
 0 0 1 1 0 0 0 0 0 1 1 1 0 0 1 1 1 1 1 0 0 1 0 1 1 1 0 1 1 1 1 1 1 1 1 1 1
 1 0 1 1 0 0 0 1 0 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 0 1 1 1 0 1 1 1 1
 1 0 1 0 0 1 1 1 1 0]


In [109]:
X=np.array(faa_list).reshape(-1, 1)
le=LabelEncoder()
y=le.fit_transform(df["sentimiento"].values)

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, stratify=y)

modelo=LogisticRegression()
modelo.fit(X_train,y_train)

y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]


print("Precisión (Accuracy):", accuracy_score(y_test, y_pred))
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

Precisión (Accuracy): 0.56

Matriz de Confusión:
[[ 0 11]
 [ 0 14]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        11
           1       0.56      1.00      0.72        14

    accuracy                           0.56        25
   macro avg       0.28      0.50      0.36        25
weighted avg       0.31      0.56      0.40        25



c:\Users\Victor\miniconda3\envs\adsp\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Victor\miniconda3\envs\adsp\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Victor\miniconda3\envs\adsp\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

In [ ]:
a=cargar_registro("REM_Turku/Data/PSG/case01_s10.edf")
 